In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt


## Загрузка данных

In [2]:
import pandas as pd


df = pd.read_csv('women-clothing-accessories.3-class.balanced.csv', sep='\t')


df.head()

,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative


In [3]:
print(f"Размер датасета: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
print(f"Распределение классов:\n{df['sentiment'].value_counts()}")

Размер датасета: (90000, 2)
Колонки: ['review', 'sentiment']
Распределение классов:
negative    30000
neautral    30000
positive    30000
Name: sentiment, dtype: int64


### Целевая переменная содержит 3 класса: negative, neautral, positive. Датасет сбалансирован.

## 1. Использовать как минимум 3 модели машинного обучения, решающие задачу классификации 

In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    df['review'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['sentiment']  # сохраняем пропорции классов
)

print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")
print(f"\nРаспределение классов в обучающей выборке:\n{y_train.value_counts()}")

Размер обучающей выборки: 72000
Размер тестовой выборки: 18000

Распределение классов в обучающей выборке:
positive    24000
negative    24000
neautral    24000
Name: sentiment, dtype: int64


### Векторизация текстов с TF-IDF

In [5]:
# Создаем векторизатор TF-IDF
tfidf_vectorizer = TfidfVectorizer(min_df=5, max_df=0.7, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Размерность матрицы признаков (обучение): {X_train_tfidf.shape}")
print(f"Размерность матрицы признаков (тест): {X_test_tfidf.shape}")
print(f"Количество признаков: {len(tfidf_vectorizer.get_feature_names_out())}")

Размерность матрицы признаков (обучение): (72000, 43640)
Размерность матрицы признаков (тест): (18000, 43640)
Количество признаков: 43640


### МОДЕЛЬ 1: Логистическая регрессия

In [6]:
lr_model = LogisticRegression(max_iter=1000, random_state=42, multi_class='ovr')
lr_model.fit(X_train_tfidf, y_train)

# Предсказания
lr_pred = lr_model.predict(X_test_tfidf)

# Метрики
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred, average='weighted')

print(f"Accuracy: {lr_accuracy:.4f}")
print(f"F1-score (weighted): {lr_f1:.4f}")

print(classification_report(y_test, lr_pred))

Accuracy: 0.7473
F1-score (weighted): 0.7476
              precision    recall  f1-score   support

    neautral       0.64      0.65      0.65      6000
    negative       0.74      0.72      0.73      6000
    positive       0.87      0.87      0.87      6000

    accuracy                           0.75     18000
   macro avg       0.75      0.75      0.75     18000
weighted avg       0.75      0.75      0.75     18000



### МОДЕЛЬ 2: Linear SVC

In [7]:
svc_model = LinearSVC(max_iter=2000, random_state=42)
svc_model.fit(X_train_tfidf, y_train)
svc_pred = svc_model.predict(X_test_tfidf)

svc_accuracy = accuracy_score(y_test, svc_pred)
svc_f1 = f1_score(y_test, svc_pred, average='weighted')

print(f"Accuracy: {svc_accuracy:.4f}")
print(f"F1-score (weighted): {svc_f1:.4f}")

print(classification_report(y_test, svc_pred))

Accuracy: 0.7295
F1-score (weighted): 0.7290
              precision    recall  f1-score   support

    neautral       0.62      0.61      0.61      6000
    negative       0.70      0.71      0.71      6000
    positive       0.86      0.87      0.87      6000

    accuracy                           0.73     18000
   macro avg       0.73      0.73      0.73     18000
weighted avg       0.73      0.73      0.73     18000



### МОДЕЛЬ 3: Random Forest

In [8]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)
rf_pred = rf_model.predict(X_test_tfidf)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1-score (weighted): {rf_f1:.4f}")

print(classification_report(y_test, rf_pred))

Accuracy: 0.7266
F1-score (weighted): 0.7266
              precision    recall  f1-score   support

    neautral       0.61      0.64      0.63      6000
    negative       0.74      0.69      0.71      6000
    positive       0.83      0.85      0.84      6000

    accuracy                           0.73     18000
   macro avg       0.73      0.73      0.73     18000
weighted avg       0.73      0.73      0.73     18000



## rusentiment_convers_distilrubert_6L

In [9]:
import torch
import gc
import subprocess

# Очищаем всё
torch.cuda.empty_cache()
gc.collect()

20

In [10]:
from deeppavlov import build_model
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd
import numpy as np




y_test = y_test.replace('neautral', 'neutral')

model = build_model('rusentiment_convers_distilrubert_6L', download=True)


neutral_idx = y_test[y_test == 'neutral'].index[:500]
negative_idx = y_test[y_test == 'negative'].index[:500]  
positive_idx = y_test[y_test == 'positive'].index[:500]

test_idx = np.concatenate([neutral_idx, negative_idx, positive_idx])
test_texts = X_test.loc[test_idx].tolist()
true_labels = y_test.loc[test_idx].tolist()

print(f"Размер выборки: {len(test_texts)}")
print(f"Истинное распределение:\n{pd.Series(true_labels).value_counts()}\n")

# Получаем предсказания модели
predictions = model(test_texts)


accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score (weighted): {f1:.4f}")

print(classification_report(true_labels, predictions))

2026-03-19 21:22:27.24 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/deeppavlov_data/classifiers/rusentiment_convers_distilrubert_6L.tar.gz download because of matching hashes
C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at DeepPavlov/distilrubert-base-cased-conversational were not used when initializing DistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_projector.weight', 'v

Размер выборки: 1500
Истинное распределение:
neutral     500
negative    500
positive    500
dtype: int64

Accuracy: 0.5640
F1-score (weighted): 0.5796
              precision    recall  f1-score   support

    negative       0.64      0.54      0.58       500
     neutral       0.42      0.64      0.51       500
    positive       0.87      0.52      0.65       500
        skip       0.00      0.00      0.00         0
      speech       0.00      0.00      0.00         0

    accuracy                           0.56      1500
   macro avg       0.39      0.34      0.35      1500
weighted avg       0.64      0.56      0.58      1500



C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## rusentiment_bert 


In [11]:
from deeppavlov import build_model
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd
import numpy as np




y_test = y_test.replace('neautral', 'neutral')

model = build_model('rusentiment_bert', download=False)



neutral_idx = y_test[y_test == 'neutral'].index[:500]
negative_idx = y_test[y_test == 'negative'].index[:500]  
positive_idx = y_test[y_test == 'positive'].index[:500]

test_idx = np.concatenate([neutral_idx, negative_idx, positive_idx])
test_texts = X_test.loc[test_idx].tolist()
true_labels = y_test.loc[test_idx].tolist()

print(f"Размер выборки: {len(test_texts)}")
print(f"Истинное распределение:\n{pd.Series(true_labels).value_counts()}\n")


predictions = model(test_texts)


filtered_predictions = []
for pred in predictions:
    if pred in ['skip', 'speech']:
        filtered_predictions.append('neutral')
    else:
        filtered_predictions.append(pred)

        

accuracy = accuracy_score(true_labels, filtered_predictions)
f1 = f1_score(true_labels, filtered_predictions, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score (weighted): {f1:.4f}")

print(classification_report(true_labels, filtered_predictions, zero_division=0))

C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenc

Размер выборки: 1500
Истинное распределение:
neutral     500
negative    500
positive    500
dtype: int64

Accuracy: 0.4987
F1-score (weighted): 0.4980
              precision    recall  f1-score   support

    negative       0.57      0.36      0.44       500
     neutral       0.38      0.50      0.43       500
    positive       0.61      0.64      0.62       500

    accuracy                           0.50      1500
   macro avg       0.52      0.50      0.50      1500
weighted avg       0.52      0.50      0.50      1500



## rusentiment_convers_distilrubert_2L

In [12]:
from deeppavlov import build_model
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd
import numpy as np


y_test = y_test.replace('neautral', 'neutral')

model = build_model('rusentiment_convers_distilrubert_2L', download=False)



neutral_idx = y_test[y_test == 'neutral'].index[:500]
negative_idx = y_test[y_test == 'negative'].index[:500]  
positive_idx = y_test[y_test == 'positive'].index[:500]

test_idx = np.concatenate([neutral_idx, negative_idx, positive_idx])
test_texts = X_test.loc[test_idx].tolist()
true_labels = y_test.loc[test_idx].tolist()

print(f"Размер выборки: {len(test_texts)}")
print(f"Истинное распределение:\n{pd.Series(true_labels).value_counts()}\n")




predictions = model(test_texts)


from collections import Counter

counter = Counter(predictions)

for label, count in counter.most_common():
    print(f"{label}: {count} ({count/len(predictions)*100:.1f}%)")


filtered_predictions = []
for pred in predictions:
    if pred in ['skip', 'speech']:
        filtered_predictions.append('neutral')
    else:
        filtered_predictions.append(pred)

        

accuracy = accuracy_score(true_labels, filtered_predictions)
f1 = f1_score(true_labels, filtered_predictions, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score (weighted): {f1:.4f}")

print(classification_report(true_labels, filtered_predictions, zero_division=0))

C:\Users\danii\anaconda3\envs\deeppavlov\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at DeepPavlov/distilrubert-tiny-cased-conversational were not used when initializing DistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_projector.bias']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to

Размер выборки: 1500
Истинное распределение:
neutral     500
negative    500
positive    500
dtype: int64

neutral: 849 (56.6%)
positive: 387 (25.8%)
negative: 241 (16.1%)
skip: 15 (1.0%)
speech: 8 (0.5%)
Accuracy: 0.5273
F1-score (weighted): 0.5233
              precision    recall  f1-score   support

    negative       0.62      0.30      0.40       500
     neutral       0.40      0.70      0.51       500
    positive       0.75      0.58      0.66       500

    accuracy                           0.53      1500
   macro avg       0.59      0.53      0.52      1500
weighted avg       0.59      0.53      0.52      1500



##  Сравнение всех моделей

| Модель | Accuracy | F1-score (weighted) | Precision (macro) | Recall (macro) |
|--------|----------|---------------------|-------------------|----------------|
| **Логистическая регрессия** | **0.7473** | **0.7476** | **0.75** | **0.75** |
| Linear SVC | 0.7295 | 0.7290 | 0.73 | 0.73 |
| Random Forest | 0.7266 | 0.7266 | 0.73 | 0.73 |
| rusentiment_convers_distilrubert_6L | 0.5640 | 0.5796 | 0.39 | 0.34 |
| rusentiment_convers_distilrubert_2L | 0.5273 | 0.5233 | 0.59 | 0.53 |
| rusentiment_bert | 0.4987 | 0.4980 | 0.52 | 0.50 |

### Вывод: Для данной задачи классические ML методы с TF-IDV векторизацией работают эффективнее готовых DeepPavlov моделей.